# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

**About**: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

### Dataset Source
The dataset is described through a Croissant schema available at:
 
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (title & description)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

Let's review the available *record sets* in this dataset. Each record set and field is referenced by its `@id` as per the Croissant schema.

**List and print all record sets with their `@id` and fields:**

In [ ]:
# Print the record sets and their fields by @id
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet: {record_set.name if hasattr(record_set, 'name') else ''} (@id={record_set.id})")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {getattr(field, 'name', '')} (@id={getattr(field, 'id', '')})")
    print()

# If you want to see a list for later reference:
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"All record set IDs: {record_set_ids}")

## 3. Data Extraction

Load data from one or more record sets into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set

# List the IDs of the record sets you want to extract. We'll extract all for demonstration.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id={record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"[!] No records found for @id={record_set_id}\n")

# For continued analysis, pick one record set: e.g., the main protein data table
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Use the first available
    print(f"\nSelected main RecordSet for analysis: {main_record_set_id}")
else:
    main_record_set_id = None

# Inspect its columns
if main_record_set_id:
    print("Columns in main record set:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate filtering, normalization, and grouping with a numeric column. You should replace the `numeric_field_id` and `group_field_id` variables below by inspecting your data columns above (these are often protein abundance or coverage percentage fields in mass spectrometry datasets).

### Filtering, Normalization, and Grouping by ID


In [ ]:
# Pick appropriate columns for demonstration.
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    print("Available columns:", df.columns.tolist())  # Review this to fill variables below
    # For demonstration, try to auto-select a numeric field (can be adjusted to your data):
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Choose the first numeric; change if needed
    else:
        # Fallback: try to find coverage or abundance fields
        for col in df.columns:
            if 'abundance' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower():
                numeric_field_id = col
                break
        else:
            numeric_field_id = df.columns[0]  # default to first

    print(f"Working with numeric field: {numeric_field_id}")

    # Make sure values are numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.5)  # Median as threshold for illustration
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered {len(filtered_df)} records where {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt grouping by a likely categorical field
    group_candidates = [c for c in df.columns if c != numeric_field_id and df[c].nunique() < len(df) // 2]
    if group_candidates:
        group_field_id = group_candidates[0]
    else:
        group_field_id = df.columns[0]  # Just for demonstration

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No main record set dataframe for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized version, and a count by group.

In [ ]:
# For plotting
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    # Distribution plot of original and normalized values
    plt.figure(figsize=(12,5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()

    if norm_col in filtered_df:
        plt.figure(figsize=(12,5))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True)
        plt.title(f"Normalized Distribution of {numeric_field_id}")
        plt.xlabel(norm_col)
        plt.show()

    # Grouped bar plot
    if group_field_id in filtered_df:
        plt.figure(figsize=(12,5))
        sns.countplot(y=filtered_df[group_field_id], order=filtered_df[group_field_id].value_counts().index[:10])
        plt.title(f"Top 10 values for {group_field_id} (filtered records)")
        plt.show()


## 6. Conclusion

In this notebook, we loaded the FAIR² dataset via its Croissant schema, explored its available record sets and fields, extracted records by their `@id`, and performed basic exploratory analysis and visualizations, referencing all entities by their Croissant `@id` as best practice. For advanced analysis, use the Croissant metadata to precisely select and interpret further fields!